In [1]:
YTDLP_PATH = "d:/projects/scripts/yt-dlp.exe"
FFMPEG_PATH = "ffmpeg"
TEMP_DIR = "../data/tmp"
CHANNELS_YAML = '../data/channels.yaml'

import yaml

with open(CHANNELS_YAML, 'r') as stream:
    try:
        channels = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        channels = {}
        print(exc)
        
from pathlib import Path

Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

handles = [sub for elem in channels.values() for sub in elem if sub not in channels['journalism']]

handles

['johnnyharris',
 'PolyMatter',
 'neoexplains',
 'SearchParty',
 'Faultlinevideos',
 'RealLifeLore',
 'AtlasPro1',
 'WendoverProductions',
 'halfasinteresting',
 'SamONellaAcademy',
 'CasuallyExplained',
 'CGPGrey',
 'InternetHistorian',
 'TierZoo',
 'billwurtz',
 'ApertureThinking',
 'Exurb1a',
 'theschooloflifetv',
 'LEMMiNO',
 'FredrikKnudsen',
 'blameitonjorge',
 'BarelySociable',
 'Nexpo',
 'RealEngineering',
 'ColdFusion',
 'MustardChannel',
 'TheB1M',
 'Kurzgesagt',
 'BobbyBroccoli',
 'TomScottGo',
 'Vsauce',
 'veritasium',
 'JacobGeller',
 'BigJoel',
 'KnowingBetter',
 'hbomberguy',
 'Shaun_vids']

In [2]:
# fetch urls parallel
def fetch_urls(channel):
    result = subprocess.run(
        [YTDLP_PATH, "--flat-playlist", "--print", "%(url)s", channel],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace"
    )
    return channel, result.stdout.strip().splitlines()

base_url = "https://www.youtube.com/"
channel_urls = ['@'.join([base_url, handle]) for handle in handles]

from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

import subprocess

# Fetch all URLs in parallel
urls = {}
with ThreadPoolExecutor() as executor:
    futures = [executor.submit(fetch_urls, ch) for ch in channel_urls]
    for future in tqdm(as_completed(futures), total=len(channel_urls), 
                       desc="Fetching channel videos"):
        channel, lines = future.result()
        print(f"Fetched {len(lines)} videos from {channel}")
        for line in lines:
            url = line.strip()
            if "/watch?v=" in url:
                video_id = url.split("v=")[1].split("&")[0]
            elif "/shorts/" in url:
                video_id = url.split("/shorts/")[1].split("?")[0]
            else:
                continue
            urls[video_id] = url

# Summary
print(f"Found {len(urls)} videos across {len(channel_urls)} channels.")

Fetching channel videos:   0%|          | 0/37 [00:00<?, ?it/s]

Fetched 175 videos from https://www.youtube.com/@PolyMatter
Fetched 42 videos from https://www.youtube.com/@InternetHistorian
Fetched 39 videos from https://www.youtube.com/@FredrikKnudsen
Fetched 92 videos from https://www.youtube.com/@CasuallyExplained
Fetched 82 videos from https://www.youtube.com/@Exurb1a
Fetched 77 videos from https://www.youtube.com/@neoexplains
Fetched 64 videos from https://www.youtube.com/@SamONellaAcademy
Fetched 103 videos from https://www.youtube.com/@Faultlinevideos
Fetched 102 videos from https://www.youtube.com/@TierZoo
Fetched 122 videos from https://www.youtube.com/@SearchParty
Fetched 194 videos from https://www.youtube.com/@CGPGrey
Fetched 165 videos from https://www.youtube.com/@AtlasPro1
Fetched 171 videos from https://www.youtube.com/@LEMMiNO
Fetched 254 videos from https://www.youtube.com/@WendoverProductions
Fetched 435 videos from https://www.youtube.com/@billwurtz
Fetched 33 videos from https://www.youtube.com/@BarelySociable
Fetched 481 video

In [3]:
# Identify incomplete stems
from pathlib import Path
from tqdm.notebook import tqdm

# Define valid formats
video_formats = {".mp4"}
image_formats = {".jpg", ".webp"}
info_formats = {".info.json"}
valid_suffixes = video_formats | image_formats | info_formats

# Build a map from stem to all corresponding file extensions
stem_to_exts = {}

for file in Path(TEMP_DIR).glob("*"):
    # Normalize compound extensions
    if file.name.endswith(".info.json"):
        stem = file.name[:-len(".info.json")]
        ext = ".info.json"
    else:
        stem = file.stem
        ext = file.suffix
    
    if stem not in stem_to_exts:
        stem_to_exts[stem] = set()
    stem_to_exts[stem].add(ext)
    
# Print the number of unique stems found
print(f"Found {len(stem_to_exts)} unique stems in the temporary directory.")

complete_stems = []
incomplete_stems = []
# Check for completeness and remove incomplete sets
for stem, exts in tqdm(stem_to_exts.items(), desc="Processing existing files"):
    has_video = any(ext in exts for ext in video_formats)
    has_thumb = any(ext in exts for ext in image_formats)
    has_info = any(ext in exts for ext in info_formats)

    if has_video and has_thumb and has_info:
        complete_stems.append(stem)
    else:            
        incomplete_stems.append(stem)

print(f"Found {len(complete_stems)} complete stem.")
print(f"Found {len(incomplete_stems)} incomplete stems.")
print()

Found 10700 unique stems in the temporary directory.


Processing existing files:   0%|          | 0/10700 [00:00<?, ?it/s]

Found 9405 complete stem.
Found 1295 incomplete stems.



In [4]:
# Removals
files_removed_count = 0
incomplete_stem_count = 0
for stem in incomplete_stems:
    for file in Path(TEMP_DIR).glob(f"{stem}*"):
        file.unlink(missing_ok=True)
        files_removed_count += 1
    incomplete_stem_count += 1

print(f"Removed {files_removed_count} files, {incomplete_stem_count} stems.")
print()

missing_data = {}
for video_id, url in urls.items():
    if video_id not in complete_stems:
        missing_data[video_id] = url
        
print(f"Found {len(missing_data)} URLs with missing data.")

Removed 2591 files, 1295 stems.

Found 3982 URLs with missing data.


In [ ]:
# Perform downloads
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess
import os
from tqdm.notebook import tqdm

def download_video(url, video_id, pbar):
    temp_out = Path(TEMP_DIR) / f"{video_id}.mp4"
    pbar.update(1)
    try:
        subprocess.run([
            YTDLP_PATH,
            "--no-playlist",
            "--download-sections", "*00:00-00:07",
            "-f", "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best",
            "-o", str(temp_out),
            "--write-info-json",
            "--write-thumbnail",
            "--no-simulate",
            url
        ], capture_output=True, text=True, encoding='utf-8', errors='replace', 
                       check=True)
        return (video_id, True)
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] {video_id}:\nSTDOUT:\n{e.stdout}\nSTDERR:\n{e.stderr}")
        return (video_id, False)

pbar = tqdm(total=len(missing_data), desc="Downloading videos", 
            unit="video")
download_status = {}
with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(download_video, url, video_id, pbar): video_id
               for video_id, url in missing_data.items()}
    for future in as_completed(futures):
        video_id, status = future.result()
        download_status[video_id] = status
pbar.close()

success_count = sum(download_status.values())
failure_count = len(download_status) - success_count

print(f"Downloaded {success_count} videos successfully.")
print(f"Failed to download {failure_count} videos.")

[ERROR] xsqjbJ6hamc:
STDOUT:
[youtube] Extracting URL: https://www.youtube.com/watch?v=xsqjbJ6hamc
[youtube] xsqjbJ6hamc: Downloading webpage
[youtube] xsqjbJ6hamc: Downloading tv client config
[youtube] xsqjbJ6hamc: Downloading tv player API JSON
[youtube] xsqjbJ6hamc: This video is age-restricted; some formats may be missing without authentication. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
[youtube] xsqjbJ6hamc: Downloading web embedded client config
[youtube] xsqjbJ6hamc: Downloading web embedded player API JSON
[youtube] xsqjbJ6hamc: Downloading ios player API JSON

STDERR:
ERROR: [youtube] xsqjbJ6hamc: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cook